# Data Preparation for Ingredient-Based Cuisine Classification

This notebook prepares the cleaned recipe dataset for later cuisine classification experiments.  
Our goal is to transform raw ingredient lists into a structured binary ingredient matrix, while preserving multi-cuisine recipes through label expansion.

## 1. Import Libraries

We first import the libraries needed for data loading, list parsing, preprocessing, train-validation splitting, and feature construction.

In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
from sklearn.model_selection import train_test_split

## 2. Load the Cleaned Dataset

We load the cleaned CSV file provided after the earlier data-cleaning stage.

In [2]:
# Load the cleaned dataset
df = pd.read_csv(r"C:\Users\willi\Desktop\NTU\Y1S2\IE0005\mini project\A\recipes_cleaned.csv")

# Inspect dataset shape and columns
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (152889, 7)
Columns: ['id', 'name', 'ingredients', 'servings', 'serving_size', 'tags', 'cuisine']


,id,name,ingredients,servings,serving_size,tags,cuisine
0,39,Biryani,"['saffron', 'milk', 'hot green chili peppers s...",6.0,1 (799 g),"['weeknight', 'time-to-make', 'course', 'main-...","['asian', 'indian']"
1,41,Carina's Tofu-Vegetable Kebabs,"['firm tofu', 'eggplant', 'small zucchini', 'r...",2.0,1 (932 g),"['weeknight', 'time-to-make', 'course', 'main-...",['north-american']
2,43,Best Blackbottom Pie,"['graham cracker crumbs', 'butter', 'milk', 'e...",8.0,1 (171 g),"['weeknight', 'time-to-make', 'course', 'cuisi...",['north-american']
3,45,Buttermilk Pie With Gingersnap Crumb Crust,"['margarine', 'egg whites', 'flour', 'buttermi...",8.0,1 (91 g),"['weeknight', 'time-to-make', 'course', 'main-...",['north-american']
4,46,A Jad - Cucumber Pickle,"['rice vinegar', 'thangkwa cucumber', 'hom dae...",1.0,1 (59 g),"['30-minutes-or-less', 'time-to-make', 'course...","['asian', 'thai']"


## 3. Keep Only the Relevant Columns

For this project, we only keep:
- `ingredients` as the feature source
- `cuisine` as the target source

In [3]:
# Keep only the columns needed for cuisine classification
df = df[["ingredients", "cuisine"]].copy()

print(df.shape)
df.head()

(152889, 2)


,ingredients,cuisine
0,"['saffron', 'milk', 'hot green chili peppers s...","['asian', 'indian']"
1,"['firm tofu', 'eggplant', 'small zucchini', 'r...",['north-american']
2,"['graham cracker crumbs', 'butter', 'milk', 'e...",['north-american']
3,"['margarine', 'egg whites', 'flour', 'buttermi...",['north-american']
4,"['rice vinegar', 'thangkwa cucumber', 'hom dae...","['asian', 'thai']"


## 4. Parse List-Like Strings into Python Lists

Because the dataset was saved in CSV format, the `ingredients` and `cuisine` columns are stored as strings that look like lists.  
We convert them back into actual Python lists for later processing.

In [4]:
def parse_list_column(x):
    """
    Convert a string representation of a list back into a Python list.
    If parsing fails or the value is missing, return an empty list.
    """
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

# Parse both list-like columns
df["ingredients"] = df["ingredients"].apply(parse_list_column)
df["cuisine"] = df["cuisine"].apply(parse_list_column)

# Quick check
print(type(df.loc[0, "ingredients"]), df.loc[0, "ingredients"][:5])
print(type(df.loc[0, "cuisine"]), df.loc[0, "cuisine"])

<class 'list'> ['saffron', 'milk', 'hot green chili peppers serranos', 'large onions', 'cloves garlic']
<class 'list'> ['asian', 'indian']


## 5. Clean and Normalize Ingredient and Cuisine Text

We apply lightweight text cleaning:
- lowercase conversion
- removal of extra spaces and formatting artifacts
- duplicate removal within each recipe

This keeps the preprocessing simple and interpretable.

In [5]:
def normalize_text(x):
    """
    Apply lightweight normalization to ingredient or cuisine text.
    This removes obvious formatting artifacts while keeping the text interpretable.
    """
    x = str(x).lower().strip()
    x = x.replace("\\", " ")
    x = x.replace('"', " ")
    x = x.replace("'", " ")
    x = re.sub(r"^[\-\s]+", "", x)
    x = re.sub(r"[\-\s]+$", "", x)
    x = re.sub(r"^\d+\s+", "", x)   # remove leading numbers such as '18 bean soup mix'
    x = re.sub(r"\s+", " ", x).strip()
    return x

def clean_ingredient_list(ingredients):
    """
    Normalize ingredient names and remove duplicates within each recipe.
    """
    cleaned = []
    seen = set()
    for ing in ingredients:
        ing = normalize_text(ing)
        if ing and ing not in seen:
            cleaned.append(ing)
            seen.add(ing)
    return cleaned

def clean_cuisine_list(cuisines):
    """
    Normalize cuisine labels and remove duplicates within each recipe.
    """
    cleaned = []
    seen = set()
    for c in cuisines:
        c = normalize_text(c)
        if c and c not in seen:
            cleaned.append(c)
            seen.add(c)
    return cleaned

# Apply cleaning
df["ingredients_clean"] = df["ingredients"].apply(clean_ingredient_list)
df["cuisine_clean"] = df["cuisine"].apply(clean_cuisine_list)

df.head()

,ingredients,cuisine,ingredients_clean,cuisine_clean
0,"[saffron, milk, hot green chili peppers serran...","[asian, indian]","[saffron, milk, hot green chili peppers serran...","[asian, indian]"
1,"[firm tofu, eggplant, small zucchini, red pepp...",[north-american],"[firm tofu, eggplant, small zucchini, red pepp...",[north-american]
2,"[graham cracker crumbs, butter, milk, egg yolk...",[north-american],"[graham cracker crumbs, butter, milk, egg yolk...",[north-american]
3,"[margarine, egg whites, flour, buttermilk, gra...",[north-american],"[margarine, egg whites, flour, buttermilk, gra...",[north-american]
4,"[rice vinegar, thangkwa cucumber, hom daeng sh...","[asian, thai]","[rice vinegar, thangkwa cucumber, hom daeng sh...","[asian, thai]"


## 6. Remove Empty Rows

Rows without ingredients or cuisine labels cannot be used for classification, so we remove them.

In [6]:
# Remove rows with empty ingredient lists or empty cuisine labels
df = df[df["ingredients_clean"].apply(len) > 0].copy()
df = df[df["cuisine_clean"].apply(len) > 0].copy()

print("Shape after removing empty rows:", df.shape)

Shape after removing empty rows: (152116, 4)


## 7. Split the Dataset Before Label Expansion

Some recipes belong to multiple cuisines.  
To avoid data leakage, we split the dataset into training and validation sets **before** expanding multi-cuisine recipes into multiple single-label rows.

In [7]:
# Split at the recipe level before exploding labels
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Recipe-level train shape:", train_df.shape)
print("Recipe-level val shape:", val_df.shape)

Recipe-level train shape: (121692, 4)
Recipe-level val shape: (30424, 4)


## 8. Expand Multi-Cuisine Recipes into Single-Label Instances

For example, if one recipe has cuisine labels `['asian', 'chinese']`,  
we convert it into two training instances:

- Recipe A → asian
- Recipe A → chinese

This allows the same ingredient set to contribute to each associated cuisine label.

In [8]:
# Expand the cuisine list so that each row has one cuisine label
train_exp = train_df.explode("cuisine_clean").reset_index(drop=True)
val_exp = val_df.explode("cuisine_clean").reset_index(drop=True)

train_exp = train_exp.rename(columns={"cuisine_clean": "cuisine_label"})
val_exp = val_exp.rename(columns={"cuisine_clean": "cuisine_label"})

print("Exploded train shape:", train_exp.shape)
print("Exploded val shape:", val_exp.shape)

# Check the number of unique cuisine labels after expansion
print("Number of cuisine labels after explosion:", train_exp["cuisine_label"].nunique())
print(train_exp["cuisine_label"].value_counts().head(20))

Exploded train shape: (187557, 4)
Exploded val shape: (46952, 4)
Number of cuisine labels after explosion: 70
cuisine_label
north-american        62352
european              34017
asian                 18671
italian               10424
mexican                9239
canadian               5369
south-west-pacific     4641
indian                 4354
australian             3471
french                 3344
african                3239
chinese                2875
greek                  2849
thai                   1781
german                 1704
south-american         1668
scandinavian           1519
spanish                1447
japanese               1399
moroccan               1022
Name: count, dtype: int64


## 9. Count Ingredient Frequency in the Training Set

Rare ingredients can make the feature space extremely sparse and noisy.  
We therefore count ingredient frequencies only on the training set.

In [9]:
# Count ingredient frequencies using only the training set
train_ingredient_counter = Counter()

for ingredients in train_exp["ingredients_clean"]:
    train_ingredient_counter.update(ingredients)

print("Unique ingredients in training set:", len(train_ingredient_counter))
print(train_ingredient_counter.most_common(20))

Unique ingredients in training set: 101147
[('olive oil', 31954), ('onion', 22819), ('garlic cloves', 15958), ('salt pepper', 15525), ('black pepper', 13581), ('lemon juice', 13047), ('water', 12538), ('cumin', 12278), ('garlic cloves minced', 11656), ('brown sugar', 11452), ('vegetable oil', 11152), ('cinnamon', 11117), ('soy sauce', 11086), ('milk', 10752), ('butter', 10511), ('salt', 10352), ('vanilla', 9892), ('baking powder', 9496), ('flour', 8903), ('tomatoes', 8792)]


## 10. Filter Rare Ingredients

We remove ingredients that appear too rarely in the training data.  
This reduces sparsity and keeps the vocabulary more manageable.

In [10]:
MIN_FREQ = 3

def filter_rare_ingredients(ingredients, counter, min_freq=3):
    """
    Keep only ingredients that appear at least min_freq times
    in the training set.
    """
    return [ing for ing in ingredients if counter[ing] >= min_freq]

# Apply filtering to both train and validation sets
train_exp["ingredients_filtered"] = train_exp["ingredients_clean"].apply(
    lambda x: filter_rare_ingredients(x, train_ingredient_counter, min_freq=MIN_FREQ)
)

val_exp["ingredients_filtered"] = val_exp["ingredients_clean"].apply(
    lambda x: filter_rare_ingredients(x, train_ingredient_counter, min_freq=MIN_FREQ)
)

## 11. Build the Ingredient Vocabulary

We construct the vocabulary only from the filtered training ingredients.

In [11]:
# Build ingredient vocabulary from the filtered training set
vocab = sorted(set(ing for row in train_exp["ingredients_filtered"] for ing in row))
ingredient_to_idx = {ing: idx for idx, ing in enumerate(vocab)}

print("Filtered vocabulary size:", len(vocab))
print(vocab[:20])

Filtered vocabulary size: 26717
['abalone mushrooms', 'abalone sauce', 'abondance cheese', 'absinthe', 'aceto balsamico', 'achiote', 'achiote annato spanish spice', 'achiote annatto', 'achiote oil achiote oil achiote oil', 'achiote paste', 'achiote powder', 'achiote seeds', 'acorn', 'acorn squash', 'acorn squash butternut squash', 'action yeast fresh yeast', 'active dry yeast', 'active dry yeast active dry yeast', 'active dry yeast dry yeast', 'active yeast']


## 12. Convert Ingredients into a Binary Presence Matrix

Each recipe is represented by a binary vector:
- 1 means the ingredient is present
- 0 means the ingredient is absent

In [12]:
def ingredients_to_multihot(ingredients, ingredient_to_idx):
    """
    Convert an ingredient list into a binary ingredient-presence vector.
    """
    vec = np.zeros(len(ingredient_to_idx), dtype=np.int8)
    for ing in ingredients:
        if ing in ingredient_to_idx:
            vec[ingredient_to_idx[ing]] = 1
    return vec

# Build train and validation feature matrices
X_train = np.vstack(
    train_exp["ingredients_filtered"].apply(
        lambda x: ingredients_to_multihot(x, ingredient_to_idx)
    ).values
)

X_val = np.vstack(
    val_exp["ingredients_filtered"].apply(
        lambda x: ingredients_to_multihot(x, ingredient_to_idx)
    ).values
)

# Target labels
y_train = train_exp["cuisine_label"].values
y_val = val_exp["cuisine_label"].values

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

X_train shape: (187557, 26717)
X_val shape: (46952, 26717)
y_train shape: (187557,)
y_val shape: (46952,)


## 13. Compare Vocabulary Size Before and After Filtering

This helps us quantify how much the rare-ingredient filtering reduced the feature space.

In [13]:
original_vocab = set(ing for row in train_exp["ingredients_clean"] for ing in row)
filtered_vocab = set(vocab)

print("Original vocab size:", len(original_vocab))
print("Filtered vocab size:", len(filtered_vocab))
print("Removed vocab size:", len(original_vocab) - len(filtered_vocab))

Original vocab size: 101147
Filtered vocab size: 26717
Removed vocab size: 74430


## 14. Save Outputs for Later Modeling

We save the processed feature matrices, labels, and vocabulary so they can be used in later classification experiments.

In [14]:
# Save matrices and labels
np.save("X_train.npy", X_train)
np.save("X_val.npy", X_val)
np.save("y_train.npy", y_train)
np.save("y_val.npy", y_val)

# Save vocabulary and processed dataframes
pd.Series(vocab).to_csv("ingredient_vocab.csv", index=False)
train_exp.to_csv("train_preprocessed_exploded.csv", index=False)
val_exp.to_csv("val_preprocessed_exploded.csv", index=False)

print("Saved preprocessing outputs.")

Saved preprocessing outputs.


## 15. Summary

At the end of preprocessing, we obtained:
- an expanded single-label version of the cuisine data
- a filtered ingredient vocabulary
- a binary ingredient presence matrix
- train and validation splits for later classification experiments

In [15]:
print("========== SUMMARY ==========")
print("Recipe-level train shape:", train_df.shape)
print("Recipe-level val shape:", val_df.shape)
print("Exploded train shape:", train_exp.shape)
print("Exploded val shape:", val_exp.shape)
print("Number of cuisine labels:", train_exp["cuisine_label"].nunique())
print("Filtered vocabulary size:", len(vocab))
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

========== SUMMARY ==========
Recipe-level train shape: (121692, 4)
Recipe-level val shape: (30424, 4)
Exploded train shape: (187557, 5)
Exploded val shape: (46952, 5)
Number of cuisine labels: 70
Filtered vocabulary size: 26717
X_train shape: (187557, 26717)
X_val shape: (46952, 26717)
